In [ ]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("sys.prefix:", sys.prefix)

# EDA & Feature Selection — Power Plant Water Consumption

**Target**: `consumption_ML` (observed annual water consumption in megaliters)  
**Source**: `Preprocessed_Power_Plant_Data.csv` (2,2251 rows × 32 columns)  

### Goals
1. Drop zero-variance and perfectly redundant columns immediately
2. Diagnose multicollinearity (VIF + correlation heatmap)
3. Understand the target distribution (log-transform check)
4. Test whether categorical variables carry statistically significant signal (Kruskal-Wallis)
5. Run a LASSO regularization path to identify which features survive shrinkage
6. Produce a justified, reduced feature set for modeling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LassoCV, lasso_path
from statsmodels.stats.outliers_influence import variance_inflation_factor

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = [12, 5]

DATA_PATH = Path('..') / 'data' / 'processed' / 'Preprocessed_Power_Plant_Data.csv'
df = pd.read_csv(DATA_PATH)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head(3)

---
## 1. Column Audit — Immediate Drops

Before any analysis, remove columns that cannot carry information:
- **Identifiers**: `plant_code`, `plant_name` — not predictive features


In [ ]:
# Zero-variance check
num_df = df.select_dtypes('number')
zero_var = num_df.columns[num_df.std() == 0].tolist()
print('Zero-variance columns:', zero_var)
if zero_var:
    print(df[zero_var].describe())
else:
    print('No zero-variance columns found.')

In [ ]:
# Cell removed during cleanup


In [ ]:
DROP_IMMEDIATELY = ['plant_code', 'plant_name']
df = df.drop(columns=DROP_IMMEDIATELY)
print(f'After immediate drops: {df.shape[1]} columns remain')
print(df.columns.tolist())

---
## 2. Target Variable Distribution

`consumption_ML` spans 0.03 to ~146,000 — extreme right skew. A log transform is almost certainly needed for linear models.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(df['consumption_ML'], bins=60, edgecolor='white')
axes[0].set_title('consumption_ML (raw)')
axes[0].set_xlabel('ML')

log_cons = np.log1p(df['consumption_ML'])
axes[1].hist(log_cons, bins=60, edgecolor='white', color='steelblue')
axes[1].set_title('log1p(consumption_ML)')
axes[1].set_xlabel('log(ML + 1)')

# QQ plot of log-transformed target
stats.probplot(log_cons, dist='norm', plot=axes[2])
axes[2].set_title('QQ plot — log1p(consumption_ML)')

plt.tight_layout()
plt.show()

print(df['consumption_ML'].describe())
print(f'\nSkewness (raw):      {df["consumption_ML"].skew():.2f}')
print(f'Skewness (log1p):    {log_cons.skew():.2f}')

In [ ]:
# Target by cooling category
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

df.boxplot(column='consumption_ML', by='cooling_category', ax=axes[0])
axes[0].set_title('consumption_ML by cooling_category')
axes[0].set_yscale('log')
axes[0].set_ylabel('ML (log scale)')

df['log_consumption'] = log_cons
df.boxplot(column='log_consumption', by='cooling_category', ax=axes[1])
axes[1].set_title('log1p(consumption_ML) by cooling_category')

plt.suptitle('')
plt.tight_layout()
plt.show()

print(df.groupby('cooling_category')['consumption_ML'].describe().round(1))

In [ ]:
# Primary scatter: net_gen_mwh vs consumption_ML
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, xscale, yscale, title in [
    (axes[0], 'linear', 'linear', 'Raw scale'),
    (axes[1], 'log',    'log',    'Log-log scale'),
]:
    for cat, grp in df.groupby('cooling_category'):
        ax.scatter(grp['net_gen_mwh'], grp['consumption_ML'], alpha=0.3, s=10, label=cat)
    ax.set_xscale(xscale)
    ax.set_yscale(yscale)
    ax.set_xlabel('net_gen_mwh')
    ax.set_ylabel('consumption_ML')
    ax.set_title(f'consumption_ML vs net_gen_mwh — {title}')
    ax.legend(title='cooling_category')
plt.tight_layout()
plt.show()

---
## 3. Correlation Heatmap

Pearson and Spearman correlations between all numeric features and the (log-transformed) target.
Known multicollinearity group to watch: `max_temp_f`, `min_temp_f`, `mean_temp_f` (r > 0.97 with each other).

In [ ]:
numeric_cols = df.select_dtypes('number').columns.tolist()
# Use log-transformed target for correlation
corr_df = df[numeric_cols].copy()
corr_df['consumption_ML'] = log_cons  # replace raw with log

pearson  = corr_df.corr(method='pearson')
spearman = corr_df.corr(method='spearman')

fig, axes = plt.subplots(1, 2, figsize=(20, 12))
for ax, mat, title in [(axes[0], pearson, 'Pearson'), (axes[1], spearman, 'Spearman')]:
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, ax=ax, square=True,
                annot_kws={'size': 7})
    ax.set_title(f'{title} Correlation')
    ax.tick_params(axis='x', rotation=45, labelsize=8)
    ax.tick_params(axis='y', rotation=0, labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Correlations with target specifically (sorted)
target_corr = pearson['consumption_ML'].drop('consumption_ML').sort_values(key=abs, ascending=False)
print('Pearson |r| with log(consumption_ML + 1):')
print(target_corr.round(3).to_string())
print()
target_corr_s = spearman['consumption_ML'].drop('consumption_ML').sort_values(key=abs, ascending=False)
print('Spearman |r| with log(consumption_ML + 1):')
print(target_corr_s.round(3).to_string())

---
## 4. Multicollinearity — Variance Inflation Factor (VIF)

VIF > 10 indicates severe multicollinearity. VIF > 5 is worth investigating.  
Expected high-VIF candidates: temp columns (r > 0.97).

In [ ]:
# VIF requires complete cases — drop rows with any NaN in numeric features
vif_cols = [c for c in numeric_cols if c not in ['consumption_ML', 'log_consumption']]
vif_data = df[vif_cols].dropna()

vif_results = pd.DataFrame({
    'feature': vif_cols,
    'VIF': [variance_inflation_factor(vif_data.values, i) for i in range(len(vif_cols))]
}).sort_values('VIF', ascending=False)

print(vif_results.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['crimson' if v > 10 else 'orange' if v > 5 else 'steelblue' for v in vif_results['VIF']]
ax.barh(vif_results['feature'], vif_results['VIF'], color=colors)
ax.axvline(10, color='crimson', linestyle='--', label='VIF=10 (severe)')
ax.axvline(5, color='orange', linestyle='--', label='VIF=5 (moderate)')
ax.set_xlabel('VIF')
ax.set_title('Variance Inflation Factor by Feature')
ax.legend()
plt.tight_layout()
plt.show()

---
## 5. Categorical Variable Significance — Kruskal-Wallis Test

Non-parametric test: does `consumption_ML` differ significantly across groups?  
H₀ = all groups have the same distribution. p < 0.05 → reject → the variable carries signal.

In [ ]:
cat_cols = ['cooling_category', 'cooling_type_eia', 'primary_fuel',
            'ashrae_zone', 'moisture_regime', 'ba_climate_zone', 'state', 'year']

results = []
for col in cat_cols:
    groups = [grp['consumption_ML'].dropna().values
              for _, grp in df.groupby(col) if len(grp) > 1]
    if len(groups) >= 2:
        stat, p = stats.kruskal(*groups)
        n_groups = df[col].nunique()
        results.append({'feature': col, 'n_groups': n_groups, 'H_stat': round(stat, 2), 'p_value': p})

kw_df = pd.DataFrame(results).sort_values('p_value')
kw_df['significant'] = kw_df['p_value'] < 0.05
print(kw_df.to_string(index=False))

In [ ]:
# Visualize top categorical variables
sig_cats = kw_df[kw_df['significant']]['feature'].tolist()[:4]
fig, axes = plt.subplots(1, len(sig_cats), figsize=(5 * len(sig_cats), 5))
if len(sig_cats) == 1:
    axes = [axes]
for ax, col in zip(axes, sig_cats):
    order = df.groupby(col)['consumption_ML'].median().sort_values(ascending=False).index
    sns.boxplot(data=df, x=col, y='log_consumption', order=order, ax=ax)
    ax.set_title(f'log(consumption_ML + 1) by {col}')
    ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

---
## 6. LASSO Regularization Path

Encode categoricals, scale features, then sweep the LASSO alpha to see which features shrink to zero first.  
Features that survive strong regularization (small alpha) are the most robustly predictive.

In [ ]:
# Build feature matrix: encode categoricals, drop remaining non-numeric
lasso_df = df.drop(columns=['consumption_ML', 'log_consumption', 'state'], errors='ignore').copy()

# Label-encode low-cardinality categoricals
encode_cols = lasso_df.select_dtypes(include=['object', 'str']).columns.tolist()
le = LabelEncoder()
for col in encode_cols:
    lasso_df[col] = le.fit_transform(lasso_df[col].astype(str))

# Drop rows with NaN
lasso_df = lasso_df.dropna()
y_idx = lasso_df.index
y = np.log1p(df.loc[y_idx, 'consumption_ML'].values)

scaler = StandardScaler()
X = scaler.fit_transform(lasso_df.values)
feature_names = lasso_df.columns.tolist()

print(f'LASSO input: {X.shape[0]} rows × {X.shape[1]} features')

In [ ]:
# LASSO path
alphas, coefs, _ = lasso_path(X, y, eps=1e-4, n_alphas=100)

fig, ax = plt.subplots(figsize=(14, 6))
for i, name in enumerate(feature_names):
    ax.plot(np.log10(alphas), coefs[i], label=name)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_xlabel('log10(alpha) — regularization strength →')
ax.set_ylabel('Coefficient')
ax.set_title('LASSO Regularization Path')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Cross-validated LASSO to find best alpha and surviving features
lasso_cv = LassoCV(cv=5, random_state=42, max_iter=10000).fit(X, y)
print(f'Best alpha (CV): {lasso_cv.alpha_:.6f}')

coef_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': lasso_cv.coef_
}).sort_values('coefficient', key=abs, ascending=False)

surviving = coef_df[coef_df['coefficient'] != 0]
zeroed    = coef_df[coef_df['coefficient'] == 0]

print(f'\nSurviving features ({len(surviving)}):')  
print(surviving.to_string(index=False))
print(f'\nZeroed out ({len(zeroed)}):')  
print(zeroed['feature'].tolist())

In [ ]:
# Bar chart of surviving coefficients
fig, ax = plt.subplots(figsize=(10, max(4, len(surviving) * 0.4)))
colors = ['steelblue' if c > 0 else 'crimson' for c in surviving['coefficient']]
ax.barh(surviving['feature'], surviving['coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('LASSO Coefficient (standardized)')
ax.set_title(f'Surviving Features at CV-optimal alpha = {lasso_cv.alpha_:.5f}')
plt.tight_layout()
plt.show()

---
## 7. Feature Reduction Summary

Combine evidence from correlation, VIF, Kruskal-Wallis, and LASSO to produce a recommended final feature set.

In [ ]:
# Summary table: correlation with target, VIF, KW significance, LASSO survived
summary = pd.DataFrame({'feature': feature_names})

# Pearson correlation with target
pearson_target = {}
for col in feature_names:
    try:
        pearson_target[col] = df[col].corr(log_cons)
    except Exception:
        pearson_target[col] = np.nan
summary['pearson_r'] = summary['feature'].map(pearson_target).round(3)

# VIF
vif_map = dict(zip(vif_results['feature'], vif_results['VIF'].round(1)))
summary['VIF'] = summary['feature'].map(vif_map)

# LASSO coefficient
lasso_map = dict(zip(coef_df['feature'], coef_df['coefficient'].round(4)))
summary['lasso_coef'] = summary['feature'].map(lasso_map)
summary['lasso_survived'] = summary['lasso_coef'] != 0

summary = summary.sort_values('lasso_survived', ascending=False)
print(summary.to_string(index=False))

In [ ]:
# Final recommended features based on all evidence
# Update this list after reviewing the outputs above
FINAL_FEATURES = surviving['feature'].tolist()
print('Recommended feature set for modeling:')
for f in FINAL_FEATURES:
    print(f'  {f}')
print(f'\nTotal: {len(FINAL_FEATURES)} features')